# Parte 3. Estabilidad de las correlaciones

**Bloque A** · **Criterio de la rúbrica:** estabilidad de correlaciones (6 pts)

**Resultado de la S1 que se valida:** correlaciones de Spearman de la sección 2.2 (Figura 3 de la S1).

| Par | ρ S1 |
| --- | --- |
| Presión sistólica – presión diastólica | 0,78 |
| Edad – presión sistólica | 0,39 |
| IMC – presión diastólica | 0,37 |
| Una o dos débiles (candidatas: colesterol – glucosa ρ = 0,03; cigarrillos – glucosa ρ = −0,09) | < 0,1 |

Requisitos de la pauta: 3 a 5 correlaciones, IC bootstrap 95 %, visualizar distribuciones, identificar
correlaciones robustas e inestables (¿el IC incluye cero?, ¿alta variabilidad?).

## Configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from utils_s2 import (
    SEED, B, ALPHA, FIG, RESULTADOS_S1, NOMBRES, NUM, COLORES,
    cargar_datos, configurar_graficos, guardar_figura, guardar_tabla,
    ic_bootstrap, p_valor_permutacion, jackknife, media_acumulada,
)

configurar_graficos()
rng = np.random.default_rng(SEED)
df = cargar_datos()
print(f"Observaciones: {df.shape[0]} | Variables: {df.shape[1]} | B = {B:,} | semilla = {SEED}")

In [ ]:
# Los datos deben reproducir las cifras de la S1
assert round(df["age"].mean(), 2) == RESULTADOS_S1["ic"]["edad_media"]["estimacion"]
assert round(df["sysBP"].mean(), 2) == RESULTADOS_S1["ic"]["sysbp_media"]["estimacion"]
assert round(df["totChol"].mean(), 2) == RESULTADOS_S1["ic"]["colesterol_media"]["estimacion"]
assert round(df["TenYearCHD"].mean() * 100, 2) == RESULTADOS_S1["ic"]["prop_chd_%"]["estimacion"]
print("OK: los datos coinciden con la Sumativa 1")

## 3.1 Bootstrap de pares

Para cada par se conservan los casos completos y se remuestrean filas completas, de modo que cada
valor de una variable permanezca emparejado con su correspondiente valor de la otra. Se generan
10.000 coeficientes de Spearman por lotes y el IC percentil al 95 % se obtiene de sus cuantiles.

In [ ]:
def rho_spearman(x, y):
    """Correlación de Spearman de un par."""
    return stats.spearmanr(x, y).statistic


def bootstrap_spearman(x, y, n_resamples=B, seed=SEED, batch=500):
    """Bootstrap pareado y por lotes para evitar matrices gigantes en memoria."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    rng_local = np.random.default_rng(seed)
    distribucion = np.empty(n_resamples)
    inicio = 0
    while inicio < n_resamples:
        m = min(batch, n_resamples - inicio)
        indices = rng_local.integers(0, n, size=(m, n))
        rx = stats.rankdata(x[indices], axis=1)
        ry = stats.rankdata(y[indices], axis=1)
        rx -= rx.mean(axis=1, keepdims=True)
        ry -= ry.mean(axis=1, keepdims=True)
        numerador = np.sum(rx * ry, axis=1)
        denominador = np.sqrt(np.sum(rx ** 2, axis=1) * np.sum(ry ** 2, axis=1))
        distribucion[inicio:inicio + m] = numerador / denominador
        inicio += m
    return distribucion


PARES = [
    ("sysBP", "diaBP"),
    ("age", "sysBP"),
    ("BMI", "diaBP"),
    ("totChol", "glucose"),
    ("cigsPerDay", "glucose"),
]

resultados_corr = []
distribuciones_corr = {}
for i, (x_nombre, y_nombre) in enumerate(PARES):
    datos = df[[x_nombre, y_nombre]].dropna()
    x = datos[x_nombre].to_numpy()
    y = datos[y_nombre].to_numpy()
    rho = rho_spearman(x, y)
    dist = bootstrap_spearman(x, y, seed=SEED + i)
    inferior, superior = np.quantile(dist, [ALPHA / 2, 1 - ALPHA / 2])
    etiqueta = f"{NOMBRES[x_nombre]} – {NOMBRES[y_nombre]}"
    distribuciones_corr[etiqueta] = dist
    resultados_corr.append({
        "Par": etiqueta, "n": len(datos), "rho": rho,
        "IC95_inf": inferior, "IC95_sup": superior,
        "Amplitud": superior - inferior, "Incluye_cero": inferior <= 0 <= superior,
    })

pd.DataFrame(resultados_corr).round(4)


## 3.2 Visualización

Distribuciones bootstrap (`p3_fig1_distribuciones_rho`) y gráfico de IC por par con línea en cero (`p3_fig2_ic_correlaciones`).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7.5))
for ax, r in zip(axes.flat, resultados_corr):
    dist = distribuciones_corr[r["Par"]]
    sns.histplot(dist, bins=45, color=COLORES["principal"], alpha=.78, ax=ax)
    ax.axvline(0, color="black", lw=1, ls=":")
    ax.axvline(r["rho"], color=COLORES["riesgo"], lw=2, label="ρ observado")
    ax.axvline(r["IC95_inf"], color=COLORES["bca"], lw=1.5, ls="--", label="IC 95 %")
    ax.axvline(r["IC95_sup"], color=COLORES["bca"], lw=1.5, ls="--")
    ax.set_title(r["Par"], fontsize=9)
    ax.set(xlabel="ρ de Spearman", ylabel="Frecuencia")
axes.flat[-1].axis("off")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False)
plt.suptitle("Estabilidad bootstrap de las correlaciones de Spearman", y=.995)
plt.tight_layout(rect=[0, .05, 1, .97])
plt.savefig(FIG / "p3_fig1_distribuciones_rho.png", dpi=200, bbox_inches="tight")
plt.show()

tabla_plot = pd.DataFrame(resultados_corr).sort_values("rho")
fig, ax = plt.subplots(figsize=(8, 4.6))
y = np.arange(len(tabla_plot))
ax.errorbar(
    tabla_plot["rho"], y,
    xerr=[tabla_plot["rho"] - tabla_plot["IC95_inf"], tabla_plot["IC95_sup"] - tabla_plot["rho"]],
    fmt="o", color=COLORES["principal"], ecolor=COLORES["bca"], capsize=4
)
ax.axvline(0, color="black", lw=1, ls=":")
ax.set_yticks(y, tabla_plot["Par"])
ax.set(xlabel="ρ de Spearman e IC bootstrap percentil 95 %", ylabel="")
ax.set_title("Correlaciones robustas e inestables")
guardar_figura("p3_fig2_ic_correlaciones")


## 3.3 Clasificación: robustas vs inestables

Tabla `p3_estabilidad_correlaciones` con ρ, IC, amplitud, ¿incluye cero? y clasificación.

In [ ]:
tabla_corr = pd.DataFrame(resultados_corr)

def clasificar(fila):
    if fila["Incluye_cero"]:
        return "Inestable: compatible con 0"
    if abs(fila["rho"]) < 0.10:
        return "Dirección estable, asociación débil"
    return "Robusta"

tabla_corr["Clasificación"] = tabla_corr.apply(clasificar, axis=1)
tabla_corr = tabla_corr.set_index("Par")
guardar_tabla(tabla_corr.round(4), "p3_estabilidad_correlaciones")


## 3.4 Interpretación

Las tres correlaciones destacadas en la S1 son robustas: sus intervalos bootstrap son estrechos y no
incluyen cero. La relación entre presión sistólica y diastólica es la más estable y fuerte;
edad–presión sistólica e IMC–presión diastólica se mantienen positivas y de magnitud moderada. Entre
las asociaciones débiles, colesterol–glucosa es inestable porque su intervalo incluye cero.
Cigarrillos–glucosa conserva una dirección negativa, pero su magnitud es pequeña: estabilidad
estadística no equivale a relevancia clínica. En consecuencia, la colinealidad entre ambas presiones
sí debe considerarse en el modelamiento de la Sumativa 3, mientras que las asociaciones débiles no
justifican por sí solas decisiones de selección de variables.